# Session 17 Exercise: Validation And Resampling

Estimate cross-validated RMSE for a housing-price model and compare it with one train/test split.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error

from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


ROOT = find_repo_root()
DATA = ROOT / "data" / "raw"


In [ ]:
housing = pd.read_csv(DATA / "housing_sales.csv")
features = ["size_sq_m", "rooms", "age_years", "renovation_score", "near_transit", "district"]
X = housing[features]
y = housing["price_k_eur"]
preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["district"]),
        ("num", "passthrough", ["size_sq_m", "rooms", "age_years", "renovation_score", "near_transit"]),
    ]
)
model = Pipeline([("preprocess", preprocess), ("model", LinearRegression())])


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error")
print(f"CV RMSE: {-cv_scores.mean():.2f}")

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(f"Holdout RMSE: {mean_squared_error(y_test, pred) ** 0.5:.2f}")


Explain why these two numbers are not expected to be identical.
